# Paired respondent assignments — ICTTE 2026 sensitivity analyses

This **independent Colab notebook** extends the original [02_operational_model.ipynb](02_operational_model.ipynb) without modifying it. The operational engine, survey coding and synthetic case generator are copied from the original notebook.

## Purpose

The reviewer recommended **using common respondent assignments across settings**. This notebook performs two paired analyses:

1. **Discount sensitivity:** the *same* respondent is assigned to each request across 35%, 45%, 55%, 65%, 75% and 85% upfront discounts, within every synthetic case × profile replication. The respondent's random within-interval WTP value is also held fixed across those discounts.
2. **WTP sensitivity:** the *same* respondent is assigned to each request for **lower**, **midpoint**, **upper** and **random** representations of interval-reported WTP. Only the WTP representation varies; no change is made to respondent identities, case, or fare discount.

The two analysis groups are independent in principle. Here both deliberately reuse the original **High-scale** seeds to allow the **75% discount + random WTP** result to be compared to the original main experiment. Five case seeds × five respondent-profile replications = 25 paired realisations per setting. Scenario A remains the same fixed homogeneous benchmark.

**Survey data:** upload the authorised encoded SPSS \`.sav\` in Colab when prompted. No respondent-level survey data are stored in this repository.

**Output:** paired long-form CSVs, summary CSVs, an assignment-audit CSV, and two figures. Re-run the analysis before updating the manuscript: results may differ from those reported for the original independent-resampling sensitivity experiments.


## 1. Setup and original simulation engine

Run all cells in order in Google Colab. Upload your own authorised `survey_encoded.sav` when prompted.


In [ ]:
!pip -q install pyreadstat

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TIME_FACTOR = 1.25

BASE_FARE = 13.0
BASE_DISTANCE_KM = 3.0
PER_KM_FARE = 2.3

SURVEY_SOLO_FARE_RMB = 30.0

# Q34 option "20 RMB or above" has no natural upper bound.
# The open-ended upper category requires an explicit sensitivity-tested cap.
Q34_OPEN_UPPER_CAP_RMB = 30.0

USE_LEGACY_TIME_SLACK_WHEN_LIMITS_MISSING = True

# Example treatment for interval-valued Q34:
# sample a value inside each reported interval.
MAIN_WTP_METHOD = "random"

# Reproducibility
BASE_SEED = 2026

print("Configuration loaded.")

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any
import math
import numpy as np
import pandas as pd

TIME_FACTOR=1.25
BASE_FARE=13.0
BASE_DISTANCE_KM=3.0
PER_KM_FARE=2.3
USE_LEGACY_TIME_SLACK_WHEN_LIMITS_MISSING=True
SURVEY_SOLO_FARE_RMB=30.0

@dataclass(frozen=True)
class Location:
    x: float; y: float

@dataclass
class PassengerProfile:
    respondent_id: Any=None
    delta: float=0.0
    omega: float=math.inf
    never_accept: bool=False
    share_willingness_score: Optional[int]=None
    q16_code: Optional[int]=None
    q29_code: Optional[int]=None
    q33_code: Optional[int]=None
    q34_code: Optional[int]=None
    wtp_survey_rmb: Optional[float]=None

@dataclass
class Request:
    request_id:int; t_r:float; o_r:Location; d_r:Location; party_size:int
    sigma:float=0.5
    max_wait:Optional[float]=None
    max_detour:Optional[float]=None
    profile:Optional[PassengerProfile]=None
    accepted:bool=False
    nominal_feasible:bool=False
    rejection_reason:Optional[str]=None
    assigned_car_id:Optional[int]=None
    offered_fare:Optional[float]=None
    offered_discount:Optional[float]=None
    delta_vkt:Optional[float]=None
    pickup_time_actual:Optional[float]=None
    dropoff_time_actual:Optional[float]=None
    actual_distance:float=0.0
    shared_distance:float=0.0
    nonshared_distance:float=0.0
    gamma:Optional[float]=None
    gamma_fare:Optional[float]=None
    final_fare:Optional[float]=None
    @property
    def direct_distance(self): return physical_distance(self.o_r,self.d_r)
    @property
    def direct_travel_time(self): return travel_time(self.o_r,self.d_r)
    @property
    def solo_fare(self): return solo_fare_from_distance(self.direct_distance)

@dataclass
class CarNode:
    location:Location; kind:str; request_id:int

@dataclass
class Car:
    car_id:int; current_location:Location; capacity:int; current_time:float=0.0
    route:List[CarNode]=field(default_factory=list)
    onboard:set=field(default_factory=set)
    pickup_times:Dict[int,float]=field(default_factory=dict)
    assigned_request_ids:set=field(default_factory=set)
    distance_travelled:float=0.0; occupied_vkt:float=0.0; passenger_km:float=0.0

@dataclass
class Candidate:
    car_id:int; route:List[CarNode]; delta_vkt:float; offer_fare:float; offer_discount:float; acceptable:bool

def physical_distance(a,b): return math.hypot(a.x-b.x,a.y-b.y)
def travel_time(a,b): return TIME_FACTOR*physical_distance(a,b)
def solo_fare_from_distance(d):
    if d<=0:return 0.0
    if d<=BASE_DISTANCE_KM:return BASE_FARE
    return BASE_FARE+(d-BASE_DISTANCE_KM)*PER_KM_FARE

def route_distance_from_state(car,route):
    total=0; cur=car.current_location
    for node in route:
        total+=physical_distance(cur,node.location);cur=node.location
    return total

def upfront_offer(request,rho):
    rho=float(rho)
    if not (0<=rho<=1): raise ValueError
    return (1-rho)*request.solo_fare, rho

def candidate_accepts(request, offer_fare, offer_discount, mode, homogeneous_params=None):
    if mode=='universal': return True
    if mode=='homogeneous':
        d=homogeneous_params['delta']; o=homogeneous_params['omega']
        return offer_discount+1e-12>=d and offer_fare<=o*request.solo_fare+1e-12
    if mode=='survey_heterogeneous':
        p=request.profile
        if p is None: raise ValueError
        if p.never_accept:return False
        return offer_discount+1e-12>=p.delta and offer_fare<=p.omega*request.solo_fare+1e-12
    raise ValueError

def simulate_route_schedule(car,route,requests):
    ct=car.current_time; cl=car.current_location
    active=set(car.onboard); pickup_times=dict(car.pickup_times); dropoff_times={}
    occ=sum(requests[r].party_size for r in active)
    if occ>car.capacity:return None
    for node in route:
        ct+=travel_time(cl,node.location);cl=node.location;req=requests[node.request_id]
        if node.kind=='pickup':
            if ct<req.t_r:ct=req.t_r
            if node.request_id in active:return None
            active.add(node.request_id);pickup_times[node.request_id]=ct
            occ=sum(requests[r].party_size for r in active)
            if occ>car.capacity:return None
        elif node.kind=='dropoff':
            if node.request_id not in active:return None
            dropoff_times[node.request_id]=ct;active.remove(node.request_id)
    for rid,dt in dropoff_times.items():
        req=requests[rid];pt=pickup_times.get(rid)
        if pt is None:return None
        wait=pt-req.t_r; pooled=dt-pt
        if req.max_wait is not None and wait>req.max_wait+1e-9:return None
        if req.max_detour is not None and pooled-req.direct_travel_time>req.max_detour+1e-9:return None
        if USE_LEGACY_TIME_SLACK_WHEN_LIMITS_MISSING and req.max_wait is None and req.max_detour is None:
            if dt-req.t_r>(1+req.sigma)*req.direct_travel_time+1e-9:return None
    return {'pickup_times':pickup_times,'dropoff_times':dropoff_times,'end_time':ct}

def generate_candidates_for_car(car,request,requests,mode,rho,hom=None):
    base=list(car.route);out=[];n=len(base)
    of,od=upfront_offer(request,rho)
    acc=candidate_accepts(request,of,od,mode,hom)
    for pp in range(n+1):
        rp=base[:pp]+[CarNode(request.o_r,'pickup',request.request_id)]+base[pp:]
        for dp in range(pp+1,len(rp)+1):
            cand=rp[:dp]+[CarNode(request.d_r,'dropoff',request.request_id)]+rp[dp:]
            if simulate_route_schedule(car,cand,requests) is None: continue
            dv=route_distance_from_state(car,cand)-route_distance_from_state(car,base)
            out.append(Candidate(car.car_id,cand,dv,of,od,acc))
    return out

def _accumulate_segment(car, moved_distance, requests):
    if moved_distance<=0:return
    active=list(car.onboard)
    onboard_people=sum(requests[r].party_size for r in active)
    car.distance_travelled += moved_distance
    car.passenger_km += moved_distance*onboard_people
    if onboard_people>0:car.occupied_vkt += moved_distance
    n_active_requests=len(active)
    for rid in active:
        req=requests[rid]
        req.actual_distance += moved_distance
        if n_active_requests>=2:req.shared_distance += moved_distance
        else:req.nonshared_distance += moved_distance

def advance_car_to_time(car,target,requests):
    eps=1e-9
    if target<car.current_time-eps:raise ValueError
    while car.current_time<target-eps:
        if not car.route:car.current_time=target;return
        node=car.route[0];dist=physical_distance(car.current_location,node.location);tt=travel_time(car.current_location,node.location);arr=car.current_time+tt
        rel=requests[node.request_id].t_r if node.kind=='pickup' else None
        if target<arr-eps:
            frac=(target-car.current_time)/tt if tt>0 else 1.0
            nl=Location(car.current_location.x+frac*(node.location.x-car.current_location.x),car.current_location.y+frac*(node.location.y-car.current_location.y))
            md=physical_distance(car.current_location,nl);_accumulate_segment(car,md,requests)
            car.current_location=nl;car.current_time=target;return
        _accumulate_segment(car,dist,requests);car.current_location=node.location;car.current_time=arr
        if rel is not None and car.current_time<rel-eps:
            if target<rel-eps:car.current_time=target;return
            car.current_time=rel
        if node.kind=='pickup':
            car.onboard.add(node.request_id);car.pickup_times[node.request_id]=car.current_time;requests[node.request_id].pickup_time_actual=car.current_time
        else:
            car.onboard.discard(node.request_id);requests[node.request_id].dropoff_time_actual=car.current_time;car.pickup_times.pop(node.request_id,None)
        car.route.pop(0)
    car.current_time=target

def finish_car_route(car,requests):
    while car.route:
        node=car.route[0];target=car.current_time+travel_time(car.current_location,node.location)
        if node.kind=='pickup':target=max(target,requests[node.request_id].t_r)
        advance_car_to_time(car,target,requests)

def gamma_settlement(req):
    if not req.accepted:return
    lt=req.actual_distance; ls=req.shared_distance; ln=req.nonshared_distance
    if lt<=1e-12:
        pg=0.0;g=0.0
    else:
        g=(ln**2)/(lt**2)
        if lt<=BASE_DISTANCE_KM:
            pg=BASE_FARE
        elif ls<=BASE_DISTANCE_KM:
            pg=BASE_FARE*g+PER_KM_FARE*(lt-BASE_DISTANCE_KM)
        else:
            pg=BASE_FARE+PER_KM_FARE*(lt-BASE_DISTANCE_KM)*g
    req.gamma=g;req.gamma_fare=pg;req.final_fare=min(req.offered_fare,pg)

def run_simulation(cars,request_list,mode='universal',rho=0.75,hom=None):
    requests={r.request_id:r for r in request_list};cd={c.car_id:c for c in cars}
    for req in sorted(request_list,key=lambda r:(r.t_r,r.request_id)):
        for car in cd.values():advance_car_to_time(car,req.t_r,requests)
        feasible=[]; acceptable=[]
        for car in cd.values():
            cs=generate_candidates_for_car(car,req,requests,mode,rho,hom);feasible+=cs;acceptable += [c for c in cs if c.acceptable]
        req.nominal_feasible=bool(feasible)
        if not feasible:req.rejection_reason='route_infeasible';continue
        if not acceptable:req.rejection_reason='passenger_unacceptable';continue
        best=min(acceptable,key=lambda c:c.delta_vkt);car=cd[best.car_id];car.route=best.route;car.assigned_request_ids.add(req.request_id)
        req.accepted=True;req.assigned_car_id=best.car_id;req.offered_fare=best.offer_fare;req.offered_discount=best.offer_discount;req.delta_vkt=best.delta_vkt
    for car in cd.values():finish_car_route(car,requests)
    for r in request_list:gamma_settlement(r)
    n=len(request_list);real=sum(r.accepted for r in request_list);nom=sum(r.nominal_feasible for r in request_list)
    total_vkt=sum(c.distance_travelled for c in cd.values());occ=sum(c.occupied_vkt for c in cd.values());pkm=sum(c.passenger_km for c in cd.values())
    offerrev=sum(r.offered_fare or 0 for r in request_list if r.accepted);finalrev=sum(r.final_fare or 0 for r in request_list if r.accepted)
    return {'summary':{'mode':mode,'rho':rho,'requests':n,'realised_services':real,'nominal_matching_rate':nom/n,'realised_service_rate':real/n,'route_infeasible_rejections':sum(r.rejection_reason=='route_infeasible' for r in request_list),'passenger_unacceptable_rejections':sum(r.rejection_reason=='passenger_unacceptable' for r in request_list),'total_vkt':total_vkt,'upfront_offer_revenue':offerrev,'final_fare_revenue':finalrev,'occupied_vkt_share':occ/total_vkt if total_vkt else np.nan,'distance_weighted_occupancy':pkm/total_vkt if total_vkt else np.nan},'requests':request_list,'cars':cd}

def build_original_case_study():
    cars=[Car(0,Location(6.7,5.6),4),Car(1,Location(66.7,10.1),4),Car(2,Location(11.5,15.5),4),Car(3,Location(2.9,15.2),3),Car(4,Location(15.5,9.3),4),Car(5,Location(50.6,36.4),4)]
    rr=[(0,(5,6.7),(28.3,6.7),1),(5,(11.8,14.3),(18.8,13.5),2),(5,(50.4,37.1),(51.6,33.5),2),(5,(4.8,7.1),(53.5,25.2),1),(5,(50.6,38),(50.2,20),2),(10,(16.9,9.7),(36.1,18.9),2),(10,(6.2,10.4),(59.4,17.3),2),(10,(60.7,9.8),(45.2,16),1),(15,(11.4,16.5),(32.3,19.8),1),(15,(13.9,7.5),(50.2,17.4),1),(15,(30.9,5.3),(42.3,23.2),1),(15,(9,10.9),(35.5,20.7),1),(15,(49.5,36.3),(32.8,32.9),2),(15,(3.5,5.5),(30.6,16.4),1),(20,(48.9,38),(28.3,13.3),2),(20,(4.8,10),(65.8,11.1),2),(20,(4.5,6.3),(31.2,36.5),1),(20,(43.3,20.2),(50.2,17.4),2),(20,(8.9,12.3),(37.2,20.9),1),(40,(62.3,16.7),(26.3,28.8),1)]
    req=[Request(i,t,Location(*o),Location(*d),k) for i,(t,o,d,k) in enumerate(rr)]
    return cars,req

## 2. Load survey data and reconstruct the original respondent profiles


In [ ]:
from google.colab import files
import pyreadstat

uploaded = files.upload()

sav_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith(".sav")
]

if not sav_files:
    raise ValueError("Please upload the encoded .sav survey file.")

sav_path = sav_files[0]

survey_raw, survey_meta = pyreadstat.read_sav(
    sav_path,
    apply_value_formats=False
)

required_cols = ["Q16", "Q29", "Q33", "Q34"]

missing = [
    col for col in required_cols
    if col not in survey_raw.columns
]

if missing:
    raise ValueError(f"Missing required survey variables: {missing}")

survey = survey_raw[required_cols].copy()

print("Survey file:", sav_path)
print("Respondents:", len(survey))
display(survey.head())

In [ ]:
def build_survey_profiles(
    survey_df,
    wtp_method=MAIN_WTP_METHOD,
    seed=BASE_SEED,
    upper_cap=Q34_OPEN_UPPER_CAP_RMB,
):
    rng = np.random.default_rng(seed)
    profiles = []

    q16_delta = {
        1: 0.25,
        2: 0.35,
        3: 0.45,
        4: 0.00,
    }

    q34_intervals = {
        1: (0.0, 5.0),
        2: (5.0, 10.0),
        3: (10.0, 20.0),
        4: (20.0, float(upper_cap)),
    }

    for respondent_id, row in survey_df.iterrows():

        q16 = int(row["Q16"])
        q29 = int(row["Q29"])
        q33 = int(row["Q33"])

        never_accept = (q16 == 4)
        delta = q16_delta[q16]

        if q33 == 2 or pd.isna(row["Q34"]):
            q34 = 0
            wtp_survey = 0.0

        else:
            q34 = int(row["Q34"])
            lower, upper = q34_intervals[q34]

            if wtp_method == "random":
                wtp_survey = float(
                    rng.uniform(lower, upper)
                )

            elif wtp_method == "midpoint":
                wtp_survey = (lower + upper) / 2

            elif wtp_method == "lower":
                wtp_survey = lower

            elif wtp_method == "upper":
                wtp_survey = upper

            else:
                raise ValueError(
                    "wtp_method must be random/midpoint/lower/upper"
                )

        omega = wtp_survey / SURVEY_SOLO_FARE_RMB

        profiles.append(
            PassengerProfile(
                respondent_id=respondent_id,
                delta=delta,
                omega=omega,
                never_accept=never_accept,
                share_willingness_score=6 - q29,
                q16_code=q16,
                q29_code=q29,
                q33_code=q33,
                q34_code=q34,
                wtp_survey_rmb=wtp_survey,
            )
        )

    return profiles


def assign_complete_profiles(
    request_list,
    profiles,
    seed=BASE_SEED,
):
    rng = np.random.default_rng(seed)

    sampled_indices = rng.integers(
        0,
        len(profiles),
        size=len(request_list)
    )

    for request, profile_index in zip(
        request_list,
        sampled_indices
    ):
        request.profile = profiles[
            profile_index
        ]

    return request_list


profiles_main = build_survey_profiles(
    survey,
    wtp_method=MAIN_WTP_METHOD,
    seed=BASE_SEED,
)

profiles_midpoint = build_survey_profiles(
    survey,
    wtp_method="midpoint",
    seed=BASE_SEED,
)

profile_df = pd.DataFrame([
    {
        "respondent_id": p.respondent_id,
        "Q16": p.q16_code,
        "delta_i": p.delta,
        "never_accept": p.never_accept,
        "Q29": p.q29_code,
        "share_willingness_score": p.share_willingness_score,
        "Q33": p.q33_code,
        "Q34": p.q34_code,
        "WTP_survey_RMB": p.wtp_survey_rmb,
        "omega_i": p.omega,
    }
    for p in profiles_main
])

display(profile_df.head())

In [ ]:
finite_delta = np.array([
    p.delta
    for p in profiles_midpoint
    if not p.never_accept
])

omega_midpoint = np.array([
    p.omega
    for p in profiles_midpoint
])

homogeneous_params = {
    "delta": float(np.median(finite_delta)),
    "omega": float(np.median(omega_midpoint)),
}

print("Scenario A homogeneous parameters:")
print(homogeneous_params)

print(
    "Equivalent median survey WTP =",
    homogeneous_params["omega"] * SURVEY_SOLO_FARE_RMB,
    "RMB"
)

# Mechanically derived break-even upfront discount for Scenario A:
# rho must satisfy rho >= delta_bar and 1-rho <= omega_bar.
ILLUSTRATIVE_RHO = max(
    homogeneous_params["delta"],
    1 - homogeneous_params["omega"],
)

print(
    "Homogeneous break-even upfront discount rho =",
    round(ILLUSTRATIVE_RHO, 4)
)

## 3. Original high-scale synthetic case generator (unchanged)


In [ ]:
def _clip_location(
    x,
    y,
    width,
    height
):
    return Location(
        float(
            np.clip(
                x,
                0.0,
                width
            )
        ),
        float(
            np.clip(
                y,
                0.0,
                height
            )
        ),
    )


def _sample_clustered_location(
    rng,
    centres,
    width,
    height,
    cluster_sd=4.5,
):
    # Sample one point around a synthetic activity centre.

    centre = centres[
        rng.integers(
            0,
            len(centres)
        )
    ]

    x = rng.normal(
        centre[0],
        cluster_sd
    )

    y = rng.normal(
        centre[1],
        cluster_sd
    )

    return _clip_location(
        x,
        y,
        width,
        height
    )


def _sample_uniform_location(
    rng,
    width,
    height,
):
    return Location(
        float(
            rng.uniform(
                0,
                width
            )
        ),
        float(
            rng.uniform(
                0,
                height
            )
        ),
    )


def build_synthetic_case(
    n_requests,
    n_vehicles,
    seed=2026,
    area_width=70.0,
    area_height=40.0,
    horizon=120.0,
    spatial_pattern="clustered",
    min_trip_distance=3.0,
):
    # Reproducible synthetic ride-pooling case.

    rng = np.random.default_rng(
        seed
    )

    centres = np.array(
        [
            [
                0.18 * area_width,
                0.22 * area_height
            ],
            [
                0.35 * area_width,
                0.72 * area_height
            ],
            [
                0.68 * area_width,
                0.30 * area_height
            ],
            [
                0.82 * area_width,
                0.78 * area_height
            ],
        ],
        dtype=float,
    )

    # -------------------------
    # Vehicles
    # -------------------------

    cars = []

    for car_id in range(
        n_vehicles
    ):

        if spatial_pattern == "clustered":

            if rng.random() < 0.5:

                location = (
                    _sample_clustered_location(
                        rng,
                        centres,
                        area_width,
                        area_height,
                        cluster_sd=6.0,
                    )
                )

            else:

                location = (
                    _sample_uniform_location(
                        rng,
                        area_width,
                        area_height,
                    )
                )

        elif spatial_pattern == "uniform":

            location = (
                _sample_uniform_location(
                    rng,
                    area_width,
                    area_height,
                )
            )

        else:

            raise ValueError(
                "spatial_pattern must be "
                "'clustered' or 'uniform'."
            )

        capacity = int(
            rng.choice(
                [
                    3,
                    4
                ],
                p=[
                    0.20,
                    0.80
                ]
            )
        )

        cars.append(
            Car(
                car_id=car_id,
                current_location=location,
                capacity=capacity,
            )
        )

    # -------------------------
    # Requests
    # -------------------------

    request_times = np.sort(
        rng.uniform(
            0.0,
            horizon,
            size=n_requests
        )
    )

    requests = []

    for request_id, request_time in enumerate(
        request_times
    ):

        if spatial_pattern == "clustered":

            origin = (
                _sample_clustered_location(
                    rng,
                    centres,
                    area_width,
                    area_height,
                    cluster_sd=4.5,
                )
            )

        else:

            origin = (
                _sample_uniform_location(
                    rng,
                    area_width,
                    area_height,
                )
            )

        destination = None

        for _ in range(
            200
        ):

            if spatial_pattern == "clustered":

                if rng.random() < 0.65:

                    destination = (
                        _sample_clustered_location(
                            rng,
                            centres,
                            area_width,
                            area_height,
                            cluster_sd=4.5,
                        )
                    )

                else:

                    destination = (
                        _clip_location(
                            rng.normal(
                                origin.x,
                                7.0
                            ),
                            rng.normal(
                                origin.y,
                                7.0
                            ),
                            area_width,
                            area_height,
                        )
                    )

            else:

                destination = (
                    _sample_uniform_location(
                        rng,
                        area_width,
                        area_height,
                    )
                )

            if (
                physical_distance(
                    origin,
                    destination
                )
                >= min_trip_distance
            ):
                break

        party_size = int(
            rng.choice(
                [
                    1,
                    2
                ],
                p=[
                    0.80,
                    0.20
                ]
            )
        )

        requests.append(
            Request(
                request_id=request_id,
                t_r=float(
                    request_time
                ),
                o_r=origin,
                d_r=destination,
                party_size=party_size,
            )
        )

    return (
        cars,
        requests
    )


# Reproducibility test
cars_syn_1, req_syn_1 = (
    build_synthetic_case(
        n_requests=50,
        n_vehicles=15,
        seed=2026,
    )
)

cars_syn_2, req_syn_2 = (
    build_synthetic_case(
        n_requests=50,
        n_vehicles=15,
        seed=2026,
    )
)

assert (
    (cars_syn_1[0].current_location.x, cars_syn_1[0].current_location.y)
    ==
    (cars_syn_2[0].current_location.x, cars_syn_2[0].current_location.y)
)

assert (
    (req_syn_1[0].o_r.x, req_syn_1[0].o_r.y)
    ==
    (req_syn_2[0].o_r.x, req_syn_2[0].o_r.y)
)

print(
    "Synthetic generator reproducibility check passed."
)

## 4. Paired-experiment configuration and helpers


In [ ]:
# Configuration: match the paper's High scale and original large-scale seeds.
N_REQUESTS = 200
N_VEHICLES = 60
CASE_SEEDS = (4101, 4102, 4103, 4104, 4105)
PROFILE_REPLICATIONS = 5
BASE_PROFILE_SEED = 80000

DISCOUNT_GRID = (0.35, 0.45, 0.55, 0.65, 0.75, 0.85)
WTP_METHODS = ("lower", "midpoint", "upper", "random")
BENCHMARK_DISCOUNT = 0.75

assert 0 <= BENCHMARK_DISCOUNT <= 1
print("High scale:", N_REQUESTS, "requests,", N_VEHICLES, "vehicles")
print("Case seeds:", CASE_SEEDS)
print("Replications per seed:", PROFILE_REPLICATIONS)
print("Derived Scenario A baseline:", homogeneous_params)
print("Paper benchmark discount:", BENCHMARK_DISCOUNT)
print("Total paired realisations per setting:", len(CASE_SEEDS) * PROFILE_REPLICATIONS)


In [ ]:
import hashlib

def profile_seed_for(case_seed, rep):
    # Identical to the existing run_repeated_synthetic_experiments.
    return BASE_PROFILE_SEED + case_seed * 100 + rep

def fresh_synthetic_case(case_seed):
    return build_synthetic_case(
        n_requests=N_REQUESTS,
        n_vehicles=N_VEHICLES,
        seed=case_seed,
        spatial_pattern="clustered",
    )

def respondent_fingerprint(requests):
    # Audit-only identifier; never export the actual survey profile values.
    ids = tuple(str(request.profile.respondent_id) for request in requests)
    digest = hashlib.sha256("|".join(ids).encode("utf-8")).hexdigest()[:16]
    return ids, digest

def run_homogeneous_case(case_seed, rho_value):
    cars, requests = fresh_synthetic_case(case_seed)
    return run_simulation(
        cars, requests, mode="homogeneous", rho=rho_value,
        hom=homogeneous_params,
    )["summary"]

def run_survey_case(case_seed, profiles, assignment_seed, rho_value):
    cars, requests = fresh_synthetic_case(case_seed)
    assign_complete_profiles(requests, profiles, seed=assignment_seed)
    identities, fingerprint = respondent_fingerprint(requests)
    summary = run_simulation(
        cars, requests, mode="survey_heterogeneous", rho=rho_value,
        hom=homogeneous_params,
    )["summary"]
    return summary, identities, fingerprint

def make_record(analysis, setting, scenario, case_seed, rep, summary, fingerprint=None):
    return {
        "analysis": analysis,
        "setting": setting,
        "scenario": scenario,
        "case_seed": case_seed,
        "profile_replication": rep,
        "profile_seed": profile_seed_for(case_seed, rep) if rep is not None else None,
        "assignment_fingerprint": fingerprint,
        "realised_service_rate": summary["realised_service_rate"],
        "realised_services": summary["realised_services"],
        "route_infeasible_rejections": summary["route_infeasible_rejections"],
        "passenger_unacceptable_rejections": summary["passenger_unacceptable_rejections"],
        "total_vkt": summary["total_vkt"],
        "final_fare_revenue": summary["final_fare_revenue"],
        "distance_weighted_occupancy": summary["distance_weighted_occupancy"],
    }


## 5. Discount sensitivity — common respondent assignments within each case/replication


In [ ]:
# Paired upfront-discount sensitivity: each case/rep keeps the identical
# respondent identity vector and the identical random WTP draw.
discount_rows = []
assignment_audit_rows = []

for case_seed in CASE_SEEDS:
    # Scenario A does not draw respondent profiles.
    for rho_value in DISCOUNT_GRID:
        summary_A = run_homogeneous_case(case_seed, rho_value)
        discount_rows.append(make_record(
            "discount", rho_value, "A Homogeneous",
            case_seed, None, summary_A,
        ))

    for rep in range(PROFILE_REPLICATIONS):
        profile_seed = profile_seed_for(case_seed, rep)
        profiles = build_survey_profiles(
            survey, wtp_method="random", seed=profile_seed,
        )
        assignment_seed = profile_seed + 1_000_000
        expected_identities = None
        expected_fingerprint = None

        for rho_value in DISCOUNT_GRID:
            summary_B, identities, fingerprint = run_survey_case(
                case_seed, profiles, assignment_seed, rho_value,
            )

            if expected_identities is None:
                expected_identities = identities
                expected_fingerprint = fingerprint
            else:
                assert identities == expected_identities, (
                    "Respondent assignments differ between discounts!"
                )

            discount_rows.append(make_record(
                "discount", rho_value, "B Survey-informed",
                case_seed, rep, summary_B, fingerprint,
            ))

        assignment_audit_rows.append({
            "analysis": "discount",
            "case_seed": case_seed,
            "profile_replication": rep,
            "profile_seed": profile_seed,
            "assignment_fingerprint": expected_fingerprint,
            "settings_compared": len(DISCOUNT_GRID),
            "assignments_identical": True,
        })

paired_discount_results = pd.DataFrame(discount_rows)
print("Discount sensitivity completed:", len(paired_discount_results), "rows")
display(
    paired_discount_results.groupby(
        ["setting", "scenario"], as_index=False
    )["realised_service_rate"].agg(["mean", "std", "count"])
)


## 6. WTP sensitivity — common respondent assignments within each case/replication


In [ ]:
# Paired WTP-interval sensitivity: fix each request's respondent identity,
# hold discount at 75%, and vary only the method for the stated WTP interval.
wtp_rows = []

for case_seed in CASE_SEEDS:
    summary_A = run_homogeneous_case(case_seed, BENCHMARK_DISCOUNT)
    wtp_rows.append(make_record(
        "wtp", "benchmark", "A Homogeneous",
        case_seed, None, summary_A,
    ))

    for rep in range(PROFILE_REPLICATIONS):
        profile_seed = profile_seed_for(case_seed, rep)
        assignment_seed = profile_seed + 1_000_000
        expected_identities = None
        expected_fingerprint = None

        for method in WTP_METHODS:
            profiles = build_survey_profiles(
                survey, wtp_method=method, seed=profile_seed,
            )
            summary_B, identities, fingerprint = run_survey_case(
                case_seed, profiles, assignment_seed, BENCHMARK_DISCOUNT,
            )

            if expected_identities is None:
                expected_identities = identities
                expected_fingerprint = fingerprint
            else:
                assert identities == expected_identities, (
                    "Respondent assignments differ between WTP methods!"
                )

            wtp_rows.append(make_record(
                "wtp", method, "B Survey-informed",
                case_seed, rep, summary_B, fingerprint,
            ))

        assignment_audit_rows.append({
            "analysis": "wtp",
            "case_seed": case_seed,
            "profile_replication": rep,
            "profile_seed": profile_seed,
            "assignment_fingerprint": expected_fingerprint,
            "settings_compared": len(WTP_METHODS),
            "assignments_identical": True,
        })

paired_wtp_results = pd.DataFrame(wtp_rows)
assignment_audit = pd.DataFrame(assignment_audit_rows)
assert assignment_audit["assignments_identical"].all()

print("WTP sensitivity completed:", len(paired_wtp_results), "rows")
print("All within-analysis respondent assignments are identical.")
display(
    paired_wtp_results.groupby(
        ["setting", "scenario"], as_index=False
    )["realised_service_rate"].agg(["mean", "std", "count"])
)


## 7. Summarise, verify cross-analysis benchmark, export CSVs


In [ ]:
# Calculate means and dispersion without claiming new results until Colab runs.
def summarise_results(df):
    return (
        df.groupby(["analysis", "setting", "scenario"], as_index=False)
          .agg(
              mean_service_rate=("realised_service_rate", "mean"),
              sd_service_rate=("realised_service_rate", "std"),
              n_runs=("realised_service_rate", "size"),
              mean_vkt=("total_vkt", "mean"),
              mean_revenue=("final_fare_revenue", "mean"),
              mean_occupancy=("distance_weighted_occupancy", "mean"),
          )
    )

paired_discount_summary = summarise_results(paired_discount_results)
paired_wtp_summary = summarise_results(paired_wtp_results)

# Explicit reproducibility cross-check: 75% + random must produce exactly
# the same runs in both analyses with the same case/profile seeds.
b_discount = paired_discount_results.loc[
    (paired_discount_results["scenario"] == "B Survey-informed")
    & (paired_discount_results["setting"] == BENCHMARK_DISCOUNT),
].sort_values(["case_seed", "profile_replication"]).reset_index(drop=True)

b_wtp = paired_wtp_results.loc[
    (paired_wtp_results["scenario"] == "B Survey-informed")
    & (paired_wtp_results["setting"] == "random"),
].sort_values(["case_seed", "profile_replication"]).reset_index(drop=True)

assert list(b_discount["assignment_fingerprint"]) == list(b_wtp["assignment_fingerprint"])
assert np.allclose(
    b_discount["realised_service_rate"].to_numpy(),
    b_wtp["realised_service_rate"].to_numpy(),
    rtol=0.0, atol=1e-12,
), "The common 75% + random-WTP benchmark is inconsistent."

display(paired_discount_summary)
display(paired_wtp_summary)

paired_discount_results.to_csv("paired_discount_results.csv", index=False)
paired_wtp_results.to_csv("paired_wtp_results.csv", index=False)
paired_discount_summary.to_csv("paired_discount_summary.csv", index=False)
paired_wtp_summary.to_csv("paired_wtp_summary.csv", index=False)
assignment_audit.to_csv("paired_assignment_audit.csv", index=False)

print("All five result/audit CSV files saved in Colab runtime.")
print("75% discount + random WTP benchmarks are identical across both analyses.")


## 8. Export sensitivity figures


In [ ]:
# Optional paper-ready figures (showing paired means, not new results until run).
fig, ax = plt.subplots(figsize=(7, 4.5))
for scenario in ("A Homogeneous", "B Survey-informed"):
    d = paired_discount_summary[
        paired_discount_summary["scenario"] == scenario
    ].sort_values("setting")
    ax.plot(d["setting"] * 100, d["mean_service_rate"] * 100,
            marker="o", label=scenario)
ax.set(xlabel="Upfront discount (%)", ylabel="Realised service rate (%)",
       title="Paired discount sensitivity — High scale")
ax.set_xticks([x * 100 for x in DISCOUNT_GRID])
ax.legend()
fig.tight_layout()
fig.savefig("paired_discount_sensitivity_high.png", dpi=300, bbox_inches="tight")
plt.show()

b = paired_wtp_summary[paired_wtp_summary["scenario"] == "B Survey-informed"]
b = b.set_index("setting").loc[list(WTP_METHODS)]
a_rate = paired_wtp_summary.loc[
    paired_wtp_summary["scenario"] == "A Homogeneous",
    "mean_service_rate",
].iloc[0]
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar([m.title() for m in WTP_METHODS], b["mean_service_rate"].to_numpy() * 100)
ax.axhline(a_rate * 100, linestyle="--", label="Scenario A benchmark")
ax.set(ylabel="Realised service rate (%)",
       title="Paired WTP sensitivity — High scale, 75% discount")
ax.legend()
fig.tight_layout()
fig.savefig("paired_wtp_sensitivity_high.png", dpi=300, bbox_inches="tight")
plt.show()


## Interpretation and reproducibility

- The common assignment applies **within** each sensitivity analysis across its settings, not across different replications.
- The benchmark is **High scale / 75% discount / random WTP** and is cross-checked across both analyses. The code uses the same original `CASE_SEEDS` and `profile_seed = 80000 + case_seed * 100 + rep`.
- Scenario A has no respondent-profile assignment; it is the homogeneous comparator, calculated once per case seed and setting.
- Within-interval WTP draws are held fixed across discount settings to prevent resampling from confounding the discount comparison.
- `paired_assignment_audit.csv` records a non-identifying SHA256 fingerprint per case/replication, and all comparisons are checked with assertions.
- **Do not copy the previous independent-resampling sensitivity percentages into the new paper results.** Rerun these paired analyses using the same authorised `.sav` input and use the exported updated results.
- This separate notebook does not alter `02_operational_model.ipynb` or any other existing file.
